# Treehouse Data Analysis Project 5: Pet Shelter Analysis
Original data from Aaron Schlegel via [Kaggle](https://www.kaggle.com/aaronschlegel/austin-animal-center-shelter-intakes-and-outcomes).

In [12]:
from datetime import datetime
import os
import math

import numpy as np
import pandas as pd

intake_file = os.path.join('data', 'aac_intakes.csv')
outcome_file = os.path.join('data', 'aac_outcomes.csv')
in_out_file = os.path.join('data', 'aac_intakes_outcomes.csv')

intakes = pd.read_csv(intake_file)
outcomes = pd.read_csv(outcome_file)
ins_outs = pd.read_csv(in_out_file)

(intakes.shape, outcomes.shape, ins_outs.shape)

((80187, 12), (80681, 12), (79672, 41))

In [67]:
# Clean data so that dates are datetime
intakes.datetime = pd.to_datetime(intakes.datetime)
intakes.datetime2 = pd.to_datetime(intakes.datetime2)

outcomes.date_of_birth = pd.to_datetime(outcomes.date_of_birth)
outcomes.datetime = pd.to_datetime(outcomes.datetime)
outcomes.monthyear = pd.to_datetime(outcomes.monthyear)


ins_outs.date_of_birth = pd.to_datetime(ins_outs.date_of_birth)
ins_outs.outcome_datetime = pd.to_datetime(ins_outs.outcome_datetime)
ins_outs.intake_datetime = pd.to_datetime(ins_outs.intake_datetime)

## Question 1

Is there an area where more pets are found?

### Answer
1. Austin, TX
2. Outside Jurisdiction
3. Travis, TX
4. 7201 Levander Loop in Austin, TX
5. Del Valle, TX

### Explanation
I grouped the intakes data by `found_location` and got the size of each group. Then I sorted the locations by group size, sorted in descending order to get the largest first, and sliced the data to get the first 5 values. I used the `intakes` data, because this is a scenario where I would think you'd want repeats to be included.

In [25]:
# group by found_location, count how many are in each location, and sort descending by the count
found_locations = intakes.groupby(by='found_location', as_index=False).size()

found_locations.sort_values(by='size', ascending=False)[:5]

## Question 2
What is the average number of pets found in a month in the year 2015? Are there months where there is a higher number of animals found?

### Answer
Average intakes per month in 2015: 1559

In general, more animals are found in the warmer months, with June, May, August, and July all being in the top 5 highest intake months.

### Explanation
I filtered the intakes table by the `datetime` column, where `datetime` was between 1/1/2015 and 12/31/2015. I used the year only for the comparison since it will default to the earliest day (i.e. 2015 = 1/1/2015 and 2016 = 1/1/2016). I used the `intakes` table because I felt this was a dataset where you'd want to include repeats.

I grouped the matching results by `datetime` with a frequency of `ME` (start of the month). This gave a series of the 12 months with an intake count for each month. 

To get the months with the highest intakes, I sorted the `Series` in descending order. Then I found the mean value of the `Series` to get the average monthly intake.

In [101]:
# average number of pets found in a month during 2015
intakes_2015 = intakes[(intakes.datetime > np.datetime64('2015')) & (intakes.datetime < np.datetime64('2016'))].groupby(pd.Grouper(key='datetime', freq='ME')).size()
# intakes_2015.sort_values(ascending=False)
intakes_2015.mean()

np.float64(1559.3333333333333)

## Question 3
What is the ratio of incoming pets vs. adopted pets?

### Answer
Approximately 2.37 more pets come into the shelter than get adopted. In other words, around 42% of pets get adopted.

### Explanation
I saved a new `DataFrame` containing pets that had an outcome of 'Adoption'. Once I had a `DataFrame` containing adopted pets only, I divided the number of intakes by the number of adoptions to get the 2.37. Flipping the order (adoptions/intakes) gave me the .42 (42%).

In [14]:
# Ratio of incoming pets vs. adopted pets
adopted_pets = ins_outs[ins_outs['outcome_type'] == 'Adoption']
len(ins_outs) / len(adopted_pets)

2.3716139786866703

## Question 4
What is the distribution of the types of animals in the shelter?

### Answer
There are significantly more dogs in the shelter than any other animal (45,366). Cats are the second most common (29,539). Other (4428) and Birds (339) are much less common.

### Explanation
I grouped the `ins_outs` data by `animal_type` and got the size of each group. This data would be better described with a chart.

In [126]:
ins_outs.groupby(by='animal_type').size()

animal_type
Bird       339
Cat      29539
Dog      45366
Other     4428
dtype: int64

## Question 5
What are the adoption rates for specific breeds? (Find the top 5 dog breeds based on count, then find the adoption percentage of each)

### Answer
1. Domestic Shorthair Mix: 42.12% adopted
2. Pit Bull Mix: 37.15% adopted
3. Chihuahua Shorthair Mix: 46.99% adopted
4. Labrador Retriever Mix: 49.45% adopted
5. Domestic Medium Hair Mix: 45.58% adopted

### Explanation
First, I grouped the `ins_outs` data by `breed` and got the size of each group (count of each breed). Then, I used the `adopted_pets` `DataFrame` from earlier, and grouped them by `breed` to get a count of how many dogs of each breed were adopted. If the breed had no adoptions, I filled the missing data with `0`.

I joined the `adoptions` DataFrame to the `breeds` DataFrame, and added a new column for `adoption_ratio`, which I caclculated by dividing the total count for a breed by the adoption count for the breed. This gave me the adoption percentage of each breed.

Finally, I sorted in descending order by `count` (total count for each breed) to get the top 5 breeds.

In [16]:
# Adoption rates for top 5 dog breeds (based on count)
breeds = ins_outs.groupby(by=['breed'], as_index=False).size().rename({'size' : 'count'}, axis=1)
adoptions = adopted_pets.groupby(by=['breed'], as_index=False).size().rename({'size': 'adopted_count'}, axis=1)
breed_outcomes = breeds.join(adoptions.set_index('breed'), on='breed')
breed_outcomes.fillna(0)

breed_outcomes['adoption_ratio'] = breed_outcomes['adopted_count'] / breed_outcomes['count'] * 100
breed_outcomes.sort_values(by='count', ascending=False)[:5]

,breed,count,adopted_count,adoption_ratio
952,Domestic Shorthair Mix,23423,9865.0,42.116723
1634,Pit Bull Mix,6256,2324.0,37.148338
670,Chihuahua Shorthair Mix,4831,2270.0,46.988201
1280,Labrador Retriever Mix,4789,2368.0,49.446649
947,Domestic Medium Hair Mix,2326,1037.0,44.582975


## Question 6
What are the adoption rates for different colorings? (Find the top 5 colorings in the shelter based on count, then find the adoption percentage of each color)

### Answer
1. Black/White - 45.20% adopted
2. Black - 40.34% adopted
3. Brown Tabby - 41.85% adopted
4. Brown - 22.06% adopted
5. White - 37.57% adopted

### Explanation
I grouped the `ins_outs` data by color and got the size of each color group. Then, I used the `adopted_pets` `DataFrame` from earlier, grouped by color, and got the count of each color group.

Then I joined the two `DataFrame`s together on `color` and filled the `NaN` values to 0. I added a column for `adoption_ratio`, and filled it with the output of `adopted_count` divided by total `count`.

Finally, I sorted in descending order and sliced off the top 5 (colors with the highest count of animals).

In [21]:
# Adoption rates for top 5 colors (based on count)
colors = ins_outs.groupby(by='color', as_index=False).size().rename({'size': 'count'}, axis=1)
color_adoptions = adopted_pets.groupby(by=['color'], as_index=False).size().rename({'size': 'adopted_count'}, axis=1)

color_outcomes = colors.join(color_adoptions.set_index('color'), on='color')
color_outcomes.fillna(0)
color_outcomes['adoption_ratio'] = color_outcomes['adopted_count'] / color_outcomes['count'] * 100

color_outcomes.sort_values(by='count', ascending=False)[:5]

,color,count,adopted_count,adoption_ratio
59,Black/White,8270,3738.0,45.199516
8,Black,6673,2692.0,40.341675
143,Brown Tabby,4471,1871.0,41.847461
123,Brown,3598,794.0,22.067815
466,White,2835,1065.0,37.566138


## Question 7
About how many animals are spayed/neutered each month? (Assume all intact males/females will be spayed/neutered)

### Answer
Average spays/neuters each month: 908

### Explanation
I filtered the `ins_outs` data for animals that hadn't been spayed/neutered on intake. Then I grouped by `intake_year` and `intake_month` to get a count of how many intact animals came into the shelter each month. 

Then I got the average of the size (monthly count) column and rounded up to the nearest whole number.

In [562]:
intacts = ins_outs[(ins_outs['sex_upon_intake'] == 'Intact Male') | (ins_outs['sex_upon_intake'] == 'Intact Female')].groupby(by=['intake_year', 'intake_month'], as_index=False).size()
intacts['size'].mean()

np.float64(907.7962962962963)

## Extra Credit 1
How many animals in the shelter are repeats? Which animal was returned to the shelter the most?

### Answer
Number of repeats: 7679
Returned most: animal_id_outcome: A721033 (9 month old Rat Terrir Mix)

### Explanation
I filtered the `ins_outs` data for animals that had an `intake_number` greater than 1. Then I used length (`len`) to get the number of repeats.

Next, I sorted in descending order by `intake_number` and sliced off the top one to get the most returned animal.

In [592]:
repeats = ins_outs[ins_outs['intake_number'] > 1]
len(repeats)
repeats.sort_values(by='intake_number', ascending=False)[:1]

,age_upon_outcome,animal_id_outcome,date_of_birth,outcome_subtype,outcome_type,sex_upon_outcome,age_upon_outcome_(days),age_upon_outcome_(years),age_upon_outcome_age_group,outcome_datetime,...,age_upon_intake_age_group,intake_datetime,intake_month,intake_year,intake_monthyear,intake_weekday,intake_hour,intake_number,time_in_shelter,time_in_shelter_days
46282,9 months,A721033,2015-05-20 00:00:00,NaN,Return to Owner,Neutered Male,270,0.739726,"(-0.025, 2.5]",2016-02-20 16:18:00,...,"(-0.025, 2.5]",2016-02-20 10:44:00,2,2016,2016-02,Saturday,10,13.0,0 days 05:34:00.000000000,0.231944


## Extra Credit 2
What are the adoption rates for the following age groups?
    
- baby: < 4 months
- young: 5 months - 2 years
- adult: 3 years - 10 years
- senior: 11+ years

### Answer
Baby: 51.17%
Young: 41.70%
Adult: 33.32%
Senior: 21.02%

### Explanation
I created a list of conditions using the `age_upon_outcome_(days)` and multiplied or divided the days of a year to get the age ranges. Since the first condition that is `True` will get that choice, I only needed 1 boolean expression per condition and omitted the senior age range. Anything that didn't fit into the provided conditions instead got the default value 'Senior'.

Then I used `np.select()` to evaluate each row in `ins_outs` and assign the choice in a new column, `age_category`, and grouped the `ins_outs` DataFrame by `age_category`, with a count for how many animals were in each.

Next, I grouped filtered the `ins_outs` data to only contain adopted pets, grouped by `age_category`, and got a count for how many animals were in each.

Finally, I joined the two `DataFrame`s together, and added a column that gave the adoption percent of each category.

In [31]:
conditions = [
    (ins_outs['age_upon_outcome_(days)'] <= 365 / 3),
    (ins_outs['age_upon_outcome_(days)'] < 365 * 3 ),
    (ins_outs['age_upon_outcome_(days)'] < 365 * 11 )
]

choices = ['Baby', 'Young', 'Adult']

ins_outs['age_category'] = np.select(conditions, choices, default='Senior')

age_categories = ins_outs.groupby(by='age_category', as_index=False).size().rename({'size': 'count'}, axis=1)
adopted_age_categories = ins_outs[ins_outs['outcome_type'] == 'Adoption'].groupby(by='age_category', as_index=False).size().rename({'size': 'adopted_count'}, axis=1)

age_outcomes = age_categories.join(adopted_age_categories.set_index('age_category'), on='age_category')
age_outcomes['adoption_ratio'] = age_outcomes['adopted_count'] / age_outcomes['count'] * 100

age_outcomes.sort_values(by='adoption_ratio', ascending=False)

,age_category,count,adopted_count,adoption_ratio
1,Baby,24618,12598,51.173938
3,Young,34794,14510,41.702592
0,Adult,18110,6034,33.318609
2,Senior,2150,452,21.023256


## Extra Credit 3
If spay/neuter for a dog costs \\\$100 and a spay/neuter for a cat costs \\\$50, how much did the shelter spend in 2015 on thse procedures?

### Answer
The shelter spent $450,800 in 2015 on spay/neuter.

### Explanation
I filtered `ins_outs` to include only dogs or cats that were intact on intake. Then I filtered that list by those with an intake year of 2015, and groupbed them by `animal_type` and got the count of each group.

Then I multiplied the number of cats by 50 and the number of dogs by 100 to get the cost of each animal_type.

Finally, I added the two numbers together to get the total spay/neuter cost for 2015.

In [692]:
intact_cats_dogs = ins_outs[
    ((ins_outs['animal_type'] == 'Dog') | (ins_outs['animal_type'] == 'Cat')) & 
    ((ins_outs['sex_upon_intake'] == 'Intact Male') | (ins_outs['sex_upon_intake'] == 'Intact Male'))
]

intact_cats_dogs_2015 = intact_cats_dogs[(intact_cats_dogs['intake_year'] == 2015)].groupby(by='animal_type').size()
intact_costs = intact_cats_dogs_2015 * [50, 100]
intact_costs['Cat'] + intact_costs['Dog']

np.int64(450800)